In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

sellers_df = spark.table(
    f"{catalog}.ecommerce_bronze.sellers"
)

sellers_df.printSchema()
sellers_df.show(5, truncate=False)

In [0]:
sellers_df.select([
    F.count(F.when(sellers_df[c].isNull(), 1)).alias(c)
    for c in sellers_df.columns
]).show()

In [0]:
silver_sellers = (
    sellers_df
    .dropDuplicates(["seller_id"])
    .withColumn("seller_id", F.trim(sellers_df["seller_id"]))
    .withColumn("seller_city", F.trim(sellers_df["seller_city"]))
    .withColumn("seller_state", F.trim(sellers_df["seller_state"]))
)

In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/sellers"

silver_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.sellers
USING DELTA
LOCATION '{silver_path}'
""")

In [0]:
spark.table(
    f"{catalog}.ecommerce_silver.sellers"
).show(5, truncate=False)